# 🛰️ Field Shift: Step 2 - Exploratory Data Analysis & Feature Engineering
**NASA Space Apps Challenge 2026**

In this notebook, we analyze the 100k+ dataset and engineer physics-informed agricultural features:
1. **Basic & Advanced EDA:** Data profiling, spatial distributions, and temporal trends.
2. **Physics-Informed Features:** Thermal Stress, Vapor Pressure Deficit (VPD).
3. **Agricultural Indicators:** Soil Sponge Index and Soil Memory (Rolling windows).

In [ ]:
# --- 1. Basic Exploratory Data Analysis (EDA) ---
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("🔍 Starting Exploratory Data Analysis (EDA)...")

# Read the raw data extracted in Step 1
df = pd.read_csv('/content/drive/MyDrive/Field_Shift_NASA_Project/Field_Shift_Perfect_100k.csv')

print(f"📊 Dataset Shape: {df.shape[0]} Rows, {df.shape[1]} Columns")
print("\nMissing Values Check (Should be 0 everywhere):")
print(df.isnull().sum())

# Set plotting style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('Field Shift - Agricultural Data Profiling across Egypt', fontsize=22, fontweight='bold', color='#2E4053')

# Plot 1: Target Variable Distribution
sns.histplot(df['Root_Soil_Moisture'], bins=50, kde=True, ax=axes[0, 0], color='#3498DB')
axes[0, 0].set_title('1. Distribution of Root Soil Moisture (Target)', fontsize=16)
axes[0, 0].set_xlabel('Moisture Level')
axes[0, 0].set_ylabel('Frequency')

# Plot 2: Land Surface Temperature (LST) by Region
sns.boxplot(data=df, x='Location_Name', y='LST_Celsius', ax=axes[0, 1], palette='YlOrRd')
axes[0, 1].set_title('2. Land Surface Temperature (LST) by Region', fontsize=16)
axes[0, 1].set_xlabel('Location')
axes[0, 1].set_ylabel('Temperature (Celsius)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Plot 3: Soil Composition
soil_comp = df.groupby('Location_Name')[['Soil_Sand_pct', 'Soil_Clay_pct']].mean().reset_index()
sample_soil = soil_comp.head(15)
sample_soil.plot(x='Location_Name', y=['Soil_Sand_pct', 'Soil_Clay_pct'],
                 kind='bar', ax=axes[1, 0], color=['#E67E22', '#7D3C98'])
axes[1, 0].set_title('3. Soil Composition (Sand vs Clay) per Region', fontsize=16)
axes[1, 0].set_xlabel('Location')
axes[1, 0].set_ylabel('Percentage (%)')
axes[1, 0].tick_params(axis='x', rotation=90)

# Plot 4: Correlation Matrix
numeric_cols = ['Temperature_Avg', 'Relative_Humidity', 'Precipitation',
                'Root_Soil_Moisture', 'NDVI', 'LST_Celsius', 'Soil_Sand_pct', 'Soil_Clay_pct']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, ax=axes[1, 1])
axes[1, 1].set_title('4. Feature Correlation Matrix', fontsize=16)

plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

In [ ]:
# --- 2. Advanced Exploratory Data Analysis & Skewness ---
print("🔍 Starting Advanced Exploratory Data Analysis (EDA)...")

# Advanced Statistical Analysis (Skewness)
print("📈 Advanced Statistical Analysis (Skewness):")
skewness = df[numeric_cols].skew().round(2)
print(skewness)
print("\n* Note: Skewness close to 0 indicates a balanced, normal distribution.\n")

fig, axes = plt.subplots(3, 1, figsize=(14, 18))
fig.suptitle('Field Shift - Advanced Data Profiling & Temporal Analysis', fontsize=22, fontweight='bold', color='#2E4053')

# Plot 1: Fixed LST Boxplot (Sample of 20 locations)
sample_locations = df['Location_Name'].unique()[:20]
df_sample_lst = df[df['Location_Name'].isin(sample_locations)]
sns.boxplot(data=df_sample_lst, x='Location_Name', y='LST_Celsius', ax=axes[0], palette='YlOrRd')
axes[0].set_title('1. Land Surface Temperature (LST) by Region (Sample of 20 Locations)', fontsize=16)
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Time-Series Trend (NDVI vs Temperature)
df['Date'] = pd.to_datetime(df['Date'])
df_time = df.groupby('Date')[['Temperature_Avg', 'NDVI']].mean().reset_index()

ax2 = axes[1].twinx()
sns.lineplot(data=df_time, x='Date', y='Temperature_Avg', ax=axes[1], color='#E74C3C', label='Avg Air Temp', linewidth=1.5)
sns.lineplot(data=df_time, x='Date', y='NDVI', ax=ax2, color='#27AE60', label='Avg NDVI (Vegetation)', linewidth=1.5)
axes[1].set_title('2. Temporal Trends: Temperature vs Vegetation Health (Jan 2025 - Oct 2026)', fontsize=16)
axes[1].set_ylabel('Temperature (Celsius)', color='#E74C3C')
ax2.set_ylabel('NDVI Index', color='#27AE60')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

# Plot 3: Scatter Plot (NDVI vs Moisture colored by LST)
df_scatter = df.sample(n=5000, random_state=42)
scatter = axes[2].scatter(df_scatter['NDVI'], df_scatter['Root_Soil_Moisture'],
                          c=df_scatter['LST_Celsius'], cmap='coolwarm', alpha=0.6, edgecolors='w', linewidth=0.5)
axes[2].set_title('3. Relationship: Plant Health (NDVI) vs Soil Moisture (colored by LST)', fontsize=16)
axes[2].set_xlabel('NDVI (Plant Health)')
axes[2].set_ylabel('Root Soil Moisture')
cbar = fig.colorbar(scatter, ax=axes[2])
cbar.set_label('LST (Celsius)')

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
fig.savefig('Field_Shift_Advanced_EDA.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3. Feature Engineering: Physics-Informed ML ---
print("🚀 Upgrading Field Shift into an Integrated Agricultural System...")

# Sort by Location and Date for accurate rolling window calculations
df = df.sort_values(by=['Location_Name', 'Date']).reset_index(drop=True)

# 1. Thermal Stress (LST - Air Temp)
df['Thermal_Stress_Diff'] = df['LST_Celsius'] - df['Temperature_Avg']

# 2. Vapor Pressure Deficit (VPD in kPa)
svp = 0.6108 * np.exp((17.27 * df['Temperature_Avg']) / (df['Temperature_Avg'] + 237.3))
df['VPD_kPa'] = svp * (1 - (df['Relative_Humidity'] / 100))

# 3. Soil Sponge Index
df['Soil_Sponge_Index'] = df['Soil_Clay_pct'] / (df['Soil_Sand_pct'] + 1e-5)

# 4. Seasonality
df['Month'] = df['Date'].dt.month
df['DayOfYear'] = df['Date'].dt.dayofyear

# 5. Soil Memory (Rolling/Lag Features) per Farm
grouped = df.groupby('Location_Name')
df['Rain_3d_sum'] = grouped['Precipitation'].transform(lambda x: x.rolling(window=3, min_periods=1).sum())
df['LST_5d_avg'] = grouped['LST_Celsius'].transform(lambda x: x.rolling(window=5, min_periods=1).mean())

df.bfill(inplace=True)

# Save the Enhanced Dataset
enhanced_path = '/content/drive/MyDrive/Field_Shift_NASA_Project/Field_Shift_Enhanced_100k.csv'
df.to_csv(enhanced_path, index=False)

print("✅ Feature Engineering Complete! The dataset is now physically and agriculturally aware.")
print(f"📊 New Shape: {df.shape[0]} Rows, {df.shape[1]} Columns")
display(df[['Date', 'Location_Name', 'Thermal_Stress_Diff', 'VPD_kPa', 'Soil_Sponge_Index', 'Rain_3d_sum']].head())